In [1]:
import pandas as pd
import altair as alt

sp_members = (
    pd.read_parquet(r'C:\Users\zkx14\Downloads\CCM_dataset\sp500list.parquet')
    .assign(
        startdt = lambda df: pd.to_datetime(df['startdt']),
        enddt = lambda df: pd.to_datetime(df['enddt'])
    )
)


In [3]:
tokeep = ['permno','date','ret','shrout','prc','sprtrn','tic']

In [4]:
CCM = (
    pd.read_parquet('CCM.parquet', columns = tokeep)
    .sort_values(by = ['permno', 'date'])
    .assign(
        mktcap = lambda df: df['prc'] * df['shrout'],
        mktcap_lag = lambda df: df.groupby('permno')['mktcap'].shift(1)
    )
)


In [ ]:
# Merge Data & Filter
SP = (
    pd.merge(CCM, sp_members, on='permno')
    .query('date.between(startdt, enddt)')
    .query('date.dt.year >= 1980')
)


In [ ]:
# Calculting Weights
SP = (SP
    .assign(
        all_mktcap = lambda df: df.groupby('date')['mktcap_lag'].transform('sum'),
        weight = lambda df: df['mktcap_lag'] / df['all_mktcap']
    )
    .sort_values(by=['date','weight'], ascending=[True, False])
)


In [ ]:
# Ranking Stocks by weight
SP['weight_rank'] = SP.groupby('date')['weight'].rank(ascending=False)


In [ ]:
# Analysing the top 10 concentration
SP_top = (SP
    .query('weight_rank <= 10')
    .groupby('date', as_index=False)
    .agg(
        Top_10 = ('weight', 'sum'),
        Top_1 = ('weight', 'first')
    )
)


In [ ]:
# Visualization
alt.Chart(SP_top, title='Total Weight of Top 10 Stocks in S&P500 Index').mark_line(color='olivedrab').encode(
        alt.X('date:T').title(None),
        alt.Y('Top_10:Q').title(None).axis(format='%').scale(zero=False)
    )


alt.Chart(...)

In [ ]:
# Advanced Chart
alt.Chart(
    SP_top.melt(id_vars='date', var_name='Group', value_name='weight'),
    title='Total Weight of Top-10 and Top-1 Stocks in S&P500 Index'
).mark_line().encode(
        alt.X('date:T').title(None),
        alt.Y('weight:Q').title(None).axis(format='%'),
        alt.Color('Group:N').scale(
            domain=['Top_10','Top_1'],
            range=['olivedrab','sandybrown']
        )
    )


alt.Chart(...)

In [ ]:
# Calculate weighted returns
SPret = (SP
    .query('weight_rank <= 10')
    .assign(w_ret = lambda df: df['ret'] * df['weight'])
    .groupby('date', as_index=False)
    .agg(
        top10_ret = ('w_ret', 'sum'),
        sp_ret = ('sprtrn', 'first')
    )
)


In [ ]:
# Compunding Annual Returns
SP_annual = (SPret
    .assign(Yr = SPret['date'].dt.year)
    .groupby('Yr', as_index=False)
    .agg(
        Top_10 = ('top10_ret', lambda s: (1 + s/100).prod() - 1),
        Index = ('sp_ret', lambda s: (1 + s/100).prod() - 1)
    )
)


In [ ]:
# Visualization
ret_plot = (
    alt.Chart(
        SP_annual.melt(
            id_vars='Yr',
            var_name='Group',
            value_name='return'
        ),
        title='Annual S&P500 Return and Its Top-10 Contribution'
    )
    .mark_area(opacity=0.5)
    .encode(
        alt.X('Yr:O').title(None).axis(format='.0f'),
        alt.Y('return:Q').title(None).axis(format='%').stack(None),
        alt.Color('Group:N').title(None)
            .scale(domain=['Index','Top_10'],
                   range=['indianRed','blue'])
    )
)

labels = (ret_plot.mark_text(dy=-5, fontWeight='bold').encode(alt.Text('return:Q').format('.0%'))
)

alt.layer(ret_plot, labels)


alt.LayerChart(...)